# CAED + RAHA Combined Evaluation

Evaluates the impact of combining CAED (gpt-5.4, single run) with RAHA on error detection performance.

Compares three configurations per dataset:
1. **RAHA alone**
2. **CAED alone** (gpt-5.4 single run from origin/main)
3. **CAED + RAHA** (union of both outputs)

In [ ]:
import io
import subprocess

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)


def annotate_errors(df_fixed: pd.DataFrame, df_with_errors: pd.DataFrame) -> pd.DataFrame:
    if df_fixed.shape != df_with_errors.shape or not all(df_fixed.columns == df_with_errors.columns):
        raise ValueError("Both dataframes must have the same structure.")
    df_fixed_str = df_fixed.astype(str)
    df_with_errors_str = df_with_errors.astype(str)
    return (df_fixed_str != df_with_errors_str).astype(int)


def calculate_metrics(true_dataset: pd.DataFrame, pred_dataset: pd.DataFrame):
    y_true = true_dataset.values.flatten()
    y_pred = pred_dataset.values.flatten()

    accuracy = accuracy_score(y_true, y_pred)

    try:
        precision = precision_score(y_true, y_pred, zero_division=0)
        recall = recall_score(y_true, y_pred, zero_division=0)
        f1 = f1_score(y_true, y_pred, zero_division=0)
    except ValueError:
        precision = recall = f1 = 0.0

    cm = confusion_matrix(y_true, y_pred)
    if cm.size == 4:
        tn, fp, fn, tp = cm.ravel()
    else:
        tn = fp = fn = tp = 0
        if len(cm) == 1:
            tn = cm[0, 0] if y_true[0] == 0 else 0
            tp = cm[0, 0] if y_true[0] == 1 else 0

    if len(set(y_true)) > 1:
        roc_auc = roc_auc_score(y_true, y_pred)
        pr_auc = average_precision_score(y_true, y_pred)
    else:
        roc_auc = None
        pr_auc = None

    return {
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "f1_score": float(f1),
        "roc_auc": float(roc_auc) if roc_auc is not None else None,
        "pr_auc": float(pr_auc) if pr_auc is not None else None,
        "true_positives_count": int(tp),
        "true_negative_count": int(tn),
        "false_positive_count": int(fp),
        "false_negative_count": int(fn),
        "predicted_positives_count": int(sum(y_pred == 1)),
        "actual_positives_count": int(sum(y_true == 1)),
        "fp_rate": float(fp / len(y_pred)),
    }

In [ ]:
# Dataset configuration
# CAED gpt-5.4 single run IDs from origin/main
DATASETS = {
    "beers": {
        "caed_run_id": "27e7cab8-83f8-4288-a21f-31b800ea30da",
        "raha_path": "./tools/raha/output/beers/annotated_output.csv",
    },
    "flights": {
        "caed_run_id": "9afd6b23-7bcd-45d8-91ad-1fa2aef8526b",
        "raha_path": "./tools/raha/output/flights/annotated_output.csv",
    },
    "hospital": {
        "caed_run_id": "443d32ab-5e1e-4d7c-87a0-d5c88db8b40f",
        "raha_path": "./tools/raha/output/hospital/annotated_output.csv",
    },
    "rayyan": {
        "caed_run_id": "fa04ddba-dfa8-42f8-8a26-25a75ab747d2",
        "raha_path": "./tools/raha/output/rayyan/annotated_output.csv",
    },
}

In [ ]:
def load_csv_from_git(run_id: str, filename: str) -> pd.DataFrame:
    """Read a CSV file from a CAED run directory in origin/main via git show."""
    git_path = f"origin/main:backend/data/{run_id}/{filename}"
    result = subprocess.run(
        ["git", "show", git_path],
        capture_output=True,
        text=True,
        cwd="..",  # run from repo root, not evaluation/
    )
    if result.returncode != 0:
        raise FileNotFoundError(f"git show failed for {git_path}: {result.stderr}")
    return pd.read_csv(io.StringIO(result.stdout))


def load_caed_from_git(run_id: str) -> pd.DataFrame:
    return load_csv_from_git(run_id, "consolidated_error_annotations.csv").astype(int)

In [ ]:
# Run evaluation for all datasets
all_results = []

for dataset, cfg in DATASETS.items():
    print(f"\n{'='*50}")
    print(f"Dataset: {dataset}")
    print(f"{'='*50}")

    run_id = cfg["caed_run_id"]

    # Load clean/dirty from the CAED run directory (consistent column names)
    clean = load_csv_from_git(run_id, "clean.csv")
    dirty = load_csv_from_git(run_id, "dirty.csv")
    ground_truth = annotate_errors(clean, dirty)

    # Load CAED output from git
    caed = load_caed_from_git(run_id)

    # Load RAHA output — align columns to CAED's names to handle minor naming differences
    raha = pd.read_csv(cfg["raha_path"]).astype(int).fillna(0)
    raha.columns = caed.columns

    # Build combined output (union)
    combined = (caed | raha).astype(int)

    # Compute metrics for each configuration
    for tool_name, pred in [("RAHA", raha), ("CAED", caed), ("CAED+RAHA", combined)]:
        m = calculate_metrics(ground_truth, pred)
        row = {"dataset": dataset, "tool": tool_name, **m}
        all_results.append(row)
        print(f"  {tool_name:12s}  precision={m['precision']:.4f}  recall={m['recall']:.4f}  f1={m['f1_score']:.4f}")

print("\nDone.")

In [ ]:
# Build summary table
results_df = pd.DataFrame(all_results)

# Pivot for easy delta calculation
pivot = results_df.pivot(index="dataset", columns="tool", values="f1_score").round(4)
pivot["Δ_f1 (CAED+RAHA vs RAHA)"] = (pivot["CAED+RAHA"] - pivot["RAHA"]).round(4)
pivot["Δ_f1 (CAED+RAHA vs CAED)"] = (pivot["CAED+RAHA"] - pivot["CAED"]).round(4)

print("\nF1-Score Comparison:")
display(pivot)

print("\nFull Metrics:")
display(results_df[["dataset", "tool", "precision", "recall", "f1_score", "accuracy",
                     "roc_auc", "pr_auc",
                     "true_positives_count", "false_positive_count", "false_negative_count",
                     "actual_positives_count", "predicted_positives_count"]].round(4))

In [ ]:
# Save results
results_df.to_csv("./raha_caed_results.csv", index=False)
print("Results saved to evaluation/raha_caed_results.csv")